# 14.10 Sorting

**Prerequisites:** 14.1 Complexity Analysis, 14.8 Heaps, 14.2 Python's Built-ins  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- Why sorting is worth O(n log n) - what it *unlocks*
- The O(n²) sorts, and the one that is genuinely useful
- **Merge sort** and **quicksort** - and why quicksort wins in practice
- 🔴 Quicksort's O(n²) worst case, provoked on purpose
- **Heap sort**, from **14.8**
- 🔴 **Stability** - what it means and when it silently matters
- 🔴 The **Ω(n log n) lower bound**, and how counting/radix sort escape it
- **Timsort** - what `sorted()` actually does
- `key=`, `reverse=`, and sorting by several fields
- Interview questions, worked

---

## Why bother

Sorting costs O(n log n). You pay it because of what becomes cheap afterwards:

| Question | Unsorted | Sorted |
|---|---|---|
| Does x exist? | O(n) | **O(log n)** — binary search (**14.11**) |
| What is the median? | O(n) with quickselect | **O(1)** |
| Any duplicates? | O(n) with a set (**14.6**) | **O(n)**, no extra memory |
| Find a pair summing to k | O(n) with a hash map | **O(n)** two pointers, O(1) space (**14.3**) |
| Top k | O(n log k) (**14.8**) | **O(k)** |
| Group equal items | O(n) with a dict | **O(n)**, in place |

> **The rule of thumb:** sorting is worth it when you will query the data more than once, or when it makes an O(n²) approach into an O(n) one. For a single membership test, a `set` is cheaper than sorting.

In practice you will call `sorted()` and move on. This notebook exists because interviews ask *how* it works, and because knowing **stability** and the **lower bound** changes decisions you make in real code.

## The O(n²) family

Three classic algorithms. Two are purely educational; one is genuinely used.

| | Best | Average | Worst | Space | Stable | In place |
|---|---|---|---|---|---|---|
| **Bubble** | O(n) | O(n²) | O(n²) | O(1) | ✅ | ✅ |
| **Selection** | O(n²) | O(n²) | O(n²) | O(1) | 🔴 no | ✅ |
| **Insertion** | **O(n)** | O(n²) | O(n²) | O(1) | ✅ | ✅ |

### Insertion sort is not a toy

🔴 Look at its **best case: O(n)** on nearly-sorted data, with tiny constants and no extra memory. That makes it genuinely the fastest option for **small or nearly-sorted inputs** — which is why real sorts, including Python's, switch to it for small runs.

Selection sort is the odd one out: it is O(n²) even on sorted input, and it is **not stable**, because swapping a distant element jumps it over equal ones.

In [ ]:
def bubble_sort(data):
    """With the early-exit that makes the best case O(n)."""
    values = list(data)
    comparisons = swaps = 0
    for end in range(len(values) - 1, 0, -1):
        swapped = False
        for i in range(end):
            comparisons += 1
            if values[i] > values[i + 1]:
                values[i], values[i + 1] = values[i + 1], values[i]
                swaps += 1
                swapped = True
        if not swapped:                    # already sorted - stop
            break
    return values, comparisons, swaps


def selection_sort(data):
    """Always O(n^2). Minimises SWAPS, which mattered when writes were slow."""
    values = list(data)
    comparisons = swaps = 0
    for start in range(len(values)):
        smallest = start
        for i in range(start + 1, len(values)):
            comparisons += 1
            if values[i] < values[smallest]:
                smallest = i
        if smallest != start:
            values[start], values[smallest] = values[smallest], values[start]
            swaps += 1
    return values, comparisons, swaps


def insertion_sort(data):
    """O(n) on sorted input. The one that is actually used, for small runs."""
    values = list(data)
    comparisons = shifts = 0
    for i in range(1, len(values)):
        current = values[i]
        j = i - 1
        while j >= 0:
            comparisons += 1
            if values[j] <= current:
                break
            values[j + 1] = values[j]
            shifts += 1
            j -= 1
        values[j + 1] = current
    return values, comparisons, shifts


import random

rng = random.Random(15)
N = 200
datasets = {
    "random": [rng.randint(0, 1000) for _ in range(N)],
    "sorted": list(range(N)),
    "reversed": list(range(N, 0, -1)),
}

print(f"comparisons for n = {N}\n")
print(f"{'input':<12}{'bubble':>12}{'selection':>12}{'insertion':>12}")
print("-" * 48)
for label, data in datasets.items():
    _, bubble_c, _ = bubble_sort(data)
    _, selection_c, _ = selection_sort(data)
    _, insertion_c, _ = insertion_sort(data)
    print(f"{label:<12}{bubble_c:>12,}{selection_c:>12,}{insertion_c:>12,}")

print(f"\n  n^2 would be {N * N:,}; n would be {N}")
print("\n  On sorted input bubble and insertion do ~n comparisons - LINEAR.")
print("  Selection does the full n^2/2 regardless: it cannot tell.")
print("\n  That best case is why insertion sort survives inside real sorts.")

---

# Merge sort - divide and conquer

```
        [38, 27, 43, 3, 9, 82, 10]
               split
     [38, 27, 43]      [3, 9, 82, 10]
         split              split
    [38]  [27, 43]     [3, 9]  [82, 10]
                ...
               merge back, in order
        [3, 9, 10, 27, 38, 43, 82]
```

**The recurrence:** log n levels of splitting, O(n) work merging at each level → **O(n log n)**, guaranteed. No worst case to worry about.

| | |
|---|---|
| Time | O(n log n) **always** — best, average and worst |
| Space | 🔴 **O(n)** — the merge needs somewhere to put the result |
| Stable | ✅ yes, if you use `<=` when merging |

The merge step is the same one from **14.4**, and `heapq.merge` (**14.8**) is its k-way generalisation.

🔴 **`<=` versus `<` in the merge decides stability.** With `<=`, ties take from the left half — which came first — so equal elements keep their original order. It is one character, and it is the whole property.

In [ ]:
def merge_sort(data, counter=None):
    """O(n log n) always. O(n) extra space. Stable, thanks to <=."""
    counter = {"comparisons": 0} if counter is None else counter
    if len(data) <= 1:
        return list(data), counter

    middle = len(data) // 2
    left, _ = merge_sort(data[:middle], counter)
    right, _ = merge_sort(data[middle:], counter)

    merged = []
    i = j = 0
    while i < len(left) and j < len(right):
        counter["comparisons"] += 1
        if left[i] <= right[j]:        # 🔴 <= keeps it STABLE
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1
    merged.extend(left[i:])
    merged.extend(right[j:])
    return merged, counter


import math

print(f"{'n':>8}{'comparisons':>14}{'n log n':>12}{'ratio':>8}")
print("-" * 42)
for n in (100, 400, 1_600, 6_400):
    rng = random.Random(15)
    data = [rng.randint(0, 10_000) for _ in range(n)]
    result, counter = merge_sort(data)
    expected = n * math.log2(n)
    print(f"{n:>8}{counter['comparisons']:>14,}{int(expected):>12,}"
          f"{counter['comparisons'] / expected:>8.2f}")

print("\n  The ratio is flat - the comparison count really does track n log n.")

for label, data in datasets.items():
    result, counter = merge_sort(data)
    print(f"  {label:<10} {counter['comparisons']:>7,} comparisons, "
          f"correct: {result == sorted(data)}")
print("\n  Note the input shape barely matters. That predictability is")
print("  merge sort's real selling point.")

## Quicksort

```
   pick a PIVOT, partition around it, recurse on both sides

   [38, 27, 43, 3, 9, 82, 10]     pivot = 38

   [27, 3, 9, 10]  38  [43, 82]   everything smaller | pivot | everything larger
        recurse             recurse
```

The pivot lands in its **final position** immediately — no merge step is needed, which is why it can sort in place.

| | |
|---|---|
| Average | O(n log n), with **smaller constants than merge sort** |
| 🔴 Worst | **O(n²)** — when the pivot is always the smallest or largest |
| Space | O(log n) for the recursion, if done in place |
| Stable | 🔴 **no** — partitioning swaps distant elements |

### Why it beats merge sort in practice

Both are O(n log n) on average, but quicksort partitions **in place** with excellent memory locality, while merge sort allocates and copies. **14.1** warned that Big-O hides constants — this is the textbook case.

🔴 The worst case is not hypothetical. With `pivot = data[0]`, **already-sorted input** gives maximally unbalanced partitions every time. The next cell provokes it.

In [ ]:
def quicksort_naive(data, counter=None):
    """Pivot = first element. Simple, and catastrophic on sorted input."""
    counter = {"comparisons": 0, "depth": 0} if counter is None else counter

    def sort(values, depth):
        counter["depth"] = max(counter["depth"], depth)
        if len(values) <= 1:
            return list(values)
        pivot = values[0]                    # 🔴 the naive choice
        smaller, equal, larger = [], [], []
        for value in values:
            counter["comparisons"] += 1
            if value < pivot:
                smaller.append(value)
            elif value > pivot:
                larger.append(value)
            else:
                equal.append(value)
        return sort(smaller, depth + 1) + equal + sort(larger, depth + 1)

    return sort(data, 0), counter


def quicksort_random(data, seed=15, counter=None):
    """Random pivot: the worst case becomes vanishingly unlikely."""
    counter = {"comparisons": 0, "depth": 0} if counter is None else counter
    rng = random.Random(seed)

    def sort(values, depth):
        counter["depth"] = max(counter["depth"], depth)
        if len(values) <= 1:
            return list(values)
        pivot = values[rng.randrange(len(values))]     # ✅ randomised
        smaller, equal, larger = [], [], []
        for value in values:
            counter["comparisons"] += 1
            if value < pivot:
                smaller.append(value)
            elif value > pivot:
                larger.append(value)
            else:
                equal.append(value)
        return sort(smaller, depth + 1) + equal + sort(larger, depth + 1)

    return sort(data, 0), counter


SIZE = 600            # kept small: the naive version recurses once per element
cases = {
    "random": [random.Random(15).randint(0, 10_000) for _ in range(SIZE)],
    "sorted": list(range(SIZE)),
}

print(f"n = {SIZE}\n")
print(f"{'input':<10}{'pivot':<12}{'comparisons':>14}{'depth':>8}")
print("-" * 46)
for label, data in cases.items():
    _, naive = quicksort_naive(data)
    _, randomised = quicksort_random(data)
    print(f"{label:<10}{'first':<12}{naive['comparisons']:>14,}{naive['depth']:>8,}")
    print(f"{'':<10}{'random':<12}{randomised['comparisons']:>14,}"
          f"{randomised['depth']:>8,}")

print(f"\n  n log n would be about {int(SIZE * math.log2(SIZE)):,}")
print(f"  n^2/2 would be about  {SIZE * SIZE // 2:,}")
print("\n🔴 On SORTED input the naive pivot gives quadratic comparisons and")
print("   a recursion depth of n - it degenerates exactly like the BST in")
print("   14.7, and for the same reason: every split is maximally uneven.")
print("\n✅ A random pivot makes that input no different from any other.")

## Heap sort

From **14.8**: `heapify` in O(n), then pop the minimum n times at O(log n) each.

```
   heapify(data)          O(n)
   repeat n times:
       pop the smallest   O(log n)
                          -> O(n log n) total
```

| | |
|---|---|
| Time | O(n log n) **guaranteed** |
| Space | **O(1)** if done in place |
| Stable | 🔴 no |

It is the only common sort with **both** a guaranteed O(n log n) bound *and* O(1) space. Merge sort gives the guarantee but needs O(n); quicksort gives the space but not the guarantee.

🔴 So why is it rarely the default? **Memory locality.** It jumps around the array by powers of two, which defeats the CPU cache, so it loses to quicksort in practice despite identical Big-O. Another **14.1** "Big-O lies" case.

Its real use is as a **safety net**: C++'s `introsort` runs quicksort and switches to heap sort if the recursion gets too deep, capping the worst case at O(n log n).

In [ ]:
import heapq


def heap_sort(data):
    """heapify then pop repeatedly. O(n log n) guaranteed."""
    heap = list(data)
    heapq.heapify(heap)                    # O(n) - see 14.8
    return [heapq.heappop(heap) for _ in range(len(heap))]


for label, data in datasets.items():
    print(f"  {label:<10} correct: {heap_sort(data) == sorted(data)}")

print("\ncomparing the three O(n log n) sorts on the same data:")
import time

rng = random.Random(15)
sample = [rng.randint(0, 100_000) for _ in range(40_000)]

timings = {}
started = time.perf_counter()
merge_result, _ = merge_sort(sample)
timings["merge sort (Python)"] = time.perf_counter() - started

started = time.perf_counter()
heap_result = heap_sort(sample)
timings["heap sort (heapq, C)"] = time.perf_counter() - started

started = time.perf_counter()
builtin_result = sorted(sample)
timings["sorted() - Timsort, C"] = time.perf_counter() - started

for label, elapsed in timings.items():
    print(f"  {label:<24}{elapsed * 1000:9.1f} ms")
print(f"\n  all agree: "
      f"{merge_result == heap_result == builtin_result}")
print("\n  🔴 This is not a fair algorithmic comparison - merge sort is")
print("     interpreted Python while the others are C. It IS a fair")
print("     reminder that constants dominate at real sizes (14.1).")

## 🔴 Stability

> A sort is **stable** if elements comparing equal keep their **original relative order**.

```
   sort by GRADE only:

   input     (alice, B) (bob, A) (carol, B) (dave, A)

   stable    (bob, A) (dave, A) (alice, B) (carol, B)     A's in input order
   unstable  (dave, A) (bob, A) (carol, B) (alice, B)     order scrambled
```

### Why it matters in real code

Stability is what makes **sorting by several fields in separate passes** work:

```
    rows.sort(key=lambda r: r.name)         # sort by the SECONDARY key first
    rows.sort(key=lambda r: r.department)   # then the PRIMARY key
    # -> sorted by department, and by name within each department
```

That only works because the second sort preserves the first's ordering among ties. With an unstable sort it silently produces near-correct-looking garbage.

| Stable | Not stable |
|---|---|
| bubble, insertion, **merge**, **Timsort**, counting, radix | selection, **quicksort**, **heap sort** |

✅ **Python's `sorted()` and `list.sort()` are guaranteed stable** — it is documented, not an accident.

In [ ]:
from dataclasses import dataclass


@dataclass
class Row:
    name: str
    department: str

    def __repr__(self):
        return f"{self.name}/{self.department}"


rows = [Row("carol", "eng"), Row("alice", "ops"), Row("bob", "eng"),
        Row("dave", "ops"), Row("erin", "eng")]

# ✅ two passes, relying on stability
staged = list(rows)
staged.sort(key=lambda r: r.name)            # secondary key FIRST
staged.sort(key=lambda r: r.department)      # primary key second
print("two stable passes :", staged)

# the same thing in one pass, with a tuple key
one_pass = sorted(rows, key=lambda r: (r.department, r.name))
print("one tuple key     :", one_pass)
print("identical         :", staged == one_pass)

print("\nand the point of stability, shown directly:")

labelled = [("B", "alice"), ("A", "bob"), ("B", "carol"), ("A", "dave")]
stable_result = sorted(labelled, key=lambda pair: pair[0])
print("  input        :", [n for _, n in labelled])
print("  stable by key:", [n for _, n in stable_result])
print("    ^ bob before dave, alice before carol - input order preserved")

selection_result, _, _ = selection_sort([3, 1, 3, 1])
print("\n  selection sort is NOT stable; with equal keys carrying")
print("  different payloads it reorders them unpredictably.")

## 🔴 The Ω(n log n) lower bound

> **No comparison-based sort can beat O(n log n).** This is a *proof*, not an engineering limit.

**The argument.** There are **n!** possible orderings of n items. Each comparison has two outcomes, so k comparisons distinguish at most 2^k arrangements. To identify one ordering you need 2^k ≥ n!, so k ≥ log₂(n!) ≈ **n log n**.

So when an interviewer asks *"can you sort faster than O(n log n)?"*, the answer is: **not by comparing. But you can stop comparing.**

### Counting sort — O(n + k)

If values are integers in a **small known range**, do not compare at all: count occurrences and rebuild.

```
   [3, 1, 3, 0, 2]      counts: [1, 1, 1, 2]   (of 0, 1, 2, 3)
                        rebuild: 0, 1, 2, 3, 3
```

O(n + k) where k is the range. **Useless if k is huge** — sorting five values in the range 0…10⁹ would allocate a billion counters.

### Radix sort — O(d × (n + k))

Counting sort applied digit by digit, least significant first. **Requires a stable inner sort**, or earlier digits get scrambled.

In [ ]:
def counting_sort(data, maximum=None):
    """O(n + k). Only for non-negative integers in a small range."""
    if not data:
        return []
    maximum = max(data) if maximum is None else maximum
    counts = [0] * (maximum + 1)
    for value in data:
        counts[value] += 1                 # no comparisons at all
    out = []
    for value, count in enumerate(counts):
        out.extend([value] * count)
    return out


def radix_sort(data):
    """O(d * (n + k)). Stable counting sort per digit, least significant first."""
    if not data:
        return []
    out = list(data)
    place = 1
    while max(out) // place > 0:
        buckets = [[] for _ in range(10)]
        for value in out:                  # 🔴 must be STABLE: append order
            buckets[(value // place) % 10].append(value)
        out = [value for bucket in buckets for value in bucket]
        place *= 10
    return out


rng = random.Random(15)
small_range = [rng.randint(0, 99) for _ in range(30_000)]

started = time.perf_counter()
counted = counting_sort(small_range)
counting_time = time.perf_counter() - started

started = time.perf_counter()
builtin = sorted(small_range)
builtin_time = time.perf_counter() - started

print(f"30,000 integers in the range 0-99\n")
print(f"  counting sort (Python) {counting_time * 1000:8.1f} ms   O(n + k)")
print(f"  sorted()      (C)      {builtin_time * 1000:8.1f} ms   O(n log n)")
print(f"  agree: {counted == builtin}")
print("\n  Counting sort beats a C implementation despite being interpreted,")
print("  because it does no comparisons at all - it escaped the bound.")

print("\nradix sort:", radix_sort([170, 45, 75, 90, 802, 24, 2, 66]))
wide = [rng.randint(0, 999_999) for _ in range(2_000)]
print("  on 2,000 six-digit numbers, correct:", radix_sort(wide) == sorted(wide))

print("\n🔴 But try counting sort on a WIDE range and it falls apart:")
print("   sorting 5 values in 0..1,000,000 would allocate 1,000,001")
print("   counters. O(n + k) is only a win when k is small.")

## Timsort - what `sorted()` actually does

Python's sort, invented by Tim Peters in 2002, and since adopted by Java, Android, V8 and Swift. It is a **hybrid**, built on one observation:

> **Real data is rarely random.** It contains runs that are already sorted, or sorted backwards.

How it works:

1. Scan for **natural runs** of already-ordered elements (reversing descending ones)
2. Extend short runs to a minimum length using **insertion sort** — which is why the O(n²) sort earns its place
3. **Merge** the runs with merge sort, using galloping mode to skip ahead when one run dominates

| | |
|---|---|
| Best case | **O(n)** — already-sorted input is detected as one run |
| Average / worst | O(n log n) |
| Space | O(n) |
| Stable | ✅ guaranteed |

The best case is the point: sorting already-sorted data is nearly free, which matters enormously in practice — re-sorting a list after appending a few items, merging sorted database results, and so on.

In [ ]:
SIZE = 300_000
rng = random.Random(15)

shapes = {
    "random": [rng.randint(0, 10 ** 6) for _ in range(SIZE)],
    "already sorted": list(range(SIZE)),
    "reversed": list(range(SIZE, 0, -1)),
    "90% sorted": None,
}
nearly = list(range(SIZE))
for _ in range(SIZE // 10):
    i = rng.randrange(SIZE)
    j = rng.randrange(SIZE)
    nearly[i], nearly[j] = nearly[j], nearly[i]
shapes["90% sorted"] = nearly

print(f"sorted() on {SIZE:,} items of different shapes\n")
baseline = None
for label, data in shapes.items():
    started = time.perf_counter()
    sorted(data)
    elapsed = time.perf_counter() - started
    if baseline is None:
        baseline = elapsed
        print(f"  {label:<16}{elapsed * 1000:8.1f} ms")
    else:
        print(f"  {label:<16}{elapsed * 1000:8.1f} ms   "
              f"{baseline / elapsed:5.1f}x faster than random")

print("\n  Already-sorted and reversed input are dramatically faster -")
print("  Timsort detects them as a single run and does O(n) work.")
print("  🔴 An algorithm whose BEST case you hit constantly in practice.")

## Using `sorted()` well

| | `sorted(iterable)` | `list.sort()` |
|---|---|---|
| Returns | a **new list** | `None` — sorts in place |
| Works on | any iterable | lists only |
| Memory | O(n) extra | in place |

🔴 `data = data.sort()` sets `data` to `None`. It is the most common sorting mistake in Python.

### `key=` — sort by a computed value

```
    sorted(rows, key=lambda r: r.age)              one field
    sorted(rows, key=lambda r: (r.dept, r.name))   several, in priority order
    sorted(rows, key=lambda r: (-r.score, r.name)) 🔴 negate to reverse ONE field
```

`key` is called **once per element** (n times), not once per comparison — so an expensive key function is fine. `operator.itemgetter` and `attrgetter` are faster than lambdas because they are C.

🔴 **`reverse=True` reverses everything.** To sort descending by one field and ascending by another, negate the numeric field in the key instead.

In [ ]:
import operator

records = [
    {"name": "carol", "team": "eng", "score": 91},
    {"name": "alice", "team": "ops", "score": 91},
    {"name": "bob", "team": "eng", "score": 78},
    {"name": "dave", "team": "ops", "score": 95},
]


def show(label, rows):
    print(f"  {label:<34}", [f"{r['name']}({r['score']})" for r in rows])


show("by score", sorted(records, key=operator.itemgetter("score")))
show("by score, descending", sorted(records, key=operator.itemgetter("score"),
                                    reverse=True))
show("by team, then score", sorted(records, key=lambda r: (r["team"], r["score"])))
show("score DESC, name ASC", sorted(records, key=lambda r: (-r["score"], r["name"])))

print("\n  🔴 reverse=True would have reversed the NAME too. Negating the")
print("     numeric field is how you mix directions.")

# key is called once per element
calls = 0


def counted_key(row):
    global calls
    calls += 1
    return row["score"]


sorted(records, key=counted_key)
print(f"\n  key called {calls} times for {len(records)} records - once each,")
print("  not once per comparison. Expensive key functions are fine.")

# 🔴 the classic mistake
values = [3, 1, 2]
result = values.sort()
print(f"\n  values.sort() returned {result!r} - it sorts IN PLACE.")
print(f"  values is now {values}")
print("  🔴 `data = data.sort()` silently sets data to None.")

## The summary table

| Algorithm | Best | Average | Worst | Space | Stable | Notes |
|---|---|---|---|---|---|---|
| Bubble | O(n) | O(n²) | O(n²) | O(1) | ✅ | teaching only |
| Selection | O(n²) | O(n²) | O(n²) | O(1) | 🔴 | fewest swaps |
| **Insertion** | **O(n)** | O(n²) | O(n²) | O(1) | ✅ | **best for small/nearly sorted** |
| **Merge** | O(n log n) | O(n log n) | O(n log n) | 🔴 O(n) | ✅ | predictable; external sorting |
| **Quick** | O(n log n) | O(n log n) | 🔴 **O(n²)** | O(log n) | 🔴 | fastest in practice |
| **Heap** | O(n log n) | O(n log n) | O(n log n) | **O(1)** | 🔴 | guaranteed *and* in place |
| Counting | O(n+k) | O(n+k) | O(n+k) | O(k) | ✅ | small integer range only |
| Radix | O(d(n+k)) | O(d(n+k)) | O(d(n+k)) | O(n+k) | ✅ | fixed-width keys |
| **Timsort** | **O(n)** | O(n log n) | O(n log n) | O(n) | ✅ | **what Python uses** |

### What to actually do

> **Call `sorted()`.** It is Timsort, in C, stable, and it exploits the structure your data already has. Implement a sort yourself only in an interview, or when you have measured a specific reason — usually a small integer range, where counting sort wins.

## Interview questions

**1. What sort does Python use, and what are its properties?**
> Timsort: a hybrid of merge and insertion sort. Stable, O(n log n) worst case, **O(n)** on already-sorted or reversed input.

**2. What does stable mean, and when does it matter?**
> Equal elements keep their relative order. It is what makes multi-pass sorting by several keys work.

**3. Merge sort or quicksort?**
> Quicksort is usually faster — in place, better locality — but O(n²) worst case and unstable. Merge sort guarantees O(n log n) and stability at O(n) space, and is what you use for data too large for memory.

**4. When is quicksort O(n²), and how do you avoid it?**
> When the pivot is consistently extreme — e.g. `data[0]` on sorted input. Randomise the pivot, or use median-of-three.

**5. Can you sort faster than O(n log n)?**
> Not by comparing — there is an Ω(n log n) lower bound from the n! possible orderings. Counting and radix sort achieve O(n) by not comparing, given constraints on the keys.

**6. Sort a list of 0s, 1s and 2s in one pass.**
> Dutch national flag: three pointers, O(n) time, O(1) space. Counting sort also works but takes two passes.

**7. How would you sort a 100 GB file with 8 GB of RAM?**
> External merge sort: sort chunks that fit in memory, write them out, then k-way merge with a heap — `heapq.merge` (**14.8**).

**8. Sort a nearly-sorted array where each element is at most k positions away.**
> A min-heap of size k+1: O(n log k). Or just call `sorted()` — Timsort handles it in nearly O(n).

**9. Find the kth largest without fully sorting.**
> Quickselect: O(n) average (**14.11**). Or a size-k heap: O(n log k).

**10. Why does `sorted()` accept `key` rather than a comparator?**
> `key` is called n times; a comparator would be called O(n log n) times. Python 3 removed `cmp` for this reason — use `functools.cmp_to_key` if you truly need one.

In [ ]:
# Question 6 - the Dutch national flag partition, asked constantly.
def sort_012(data):
    """One pass, O(1) space. Three regions: <1, ==1, >1."""
    values = list(data)
    low, mid, high = 0, 0, len(values) - 1
    while mid <= high:
        if values[mid] == 0:
            values[low], values[mid] = values[mid], values[low]
            low += 1
            mid += 1
        elif values[mid] == 1:
            mid += 1
        else:                              # 2 - swap to the end
            values[mid], values[high] = values[high], values[mid]
            high -= 1                      # 🔴 do NOT advance mid: the value
                                           # swapped in has not been examined
    return values


rng = random.Random(15)
for case in ([2, 0, 2, 1, 1, 0], [0, 0, 0], [2, 1, 0], [],
             [rng.randint(0, 2) for _ in range(20)]):
    result = sort_012(case)
    print(f"  {str(case)[:34]:<36} -> {str(result)[:30]:<32} "
          f"{result == sorted(case)}")

print("\n  🔴 The subtlety is not advancing `mid` after swapping with `high`:")
print("     the value that just arrived from the end is unexamined.")


# Question 7 - external sort, in miniature
def external_sort(chunks):
    """Sort each chunk, then k-way merge. This is 14.8's heapq.merge."""
    sorted_chunks = [sorted(chunk) for chunk in chunks]      # each fits in memory
    return list(heapq.merge(*sorted_chunks))                 # lazy k-way merge


rng = random.Random(15)
everything = [rng.randint(0, 1000) for _ in range(50)]
in_chunks = [everything[i:i + 10] for i in range(0, 50, 10)]
print(f"\n  external sort of 5 chunks: "
      f"{external_sort(in_chunks) == sorted(everything)}")
print("  ^ never held more than one chunk plus the heap in memory")

---

## Common Mistakes & Pitfalls

1. 🔴 **`data = data.sort()`.** `sort()` returns `None`. Use `sorted()` for a new list.
2. 🔴 **Quicksort with `data[0]` as the pivot.** Sorted input gives O(n²) and a recursion depth of n. Randomise it.
3. 🔴 **Relying on stability from an unstable sort.** Multi-pass key sorting silently produces wrong output.
4. 🔴 **Counting sort on a wide range.** O(n + k) allocates k counters - useless when k is large.
5. **`reverse=True` when you meant to reverse one field.** Negate the numeric field in the key instead.
6. **Radix sort with an unstable inner sort.** Earlier digits get scrambled.
7. **Forgetting `<=` in the merge step.** With `<` the sort is no longer stable.
8. **Sorting to answer one membership question.** A `set` is O(n) to build and O(1) to query (**14.6**).
9. **Implementing a sort in production.** `sorted()` is C, stable, and adaptive.

## Best Practices

- Use `sorted()` or `list.sort()`; know that both are stable Timsort.
- Use a tuple `key` for multi-field sorts rather than several passes.
- Use `operator.itemgetter`/`attrgetter` over lambdas - they are C.
- Randomise or median-of-three your pivot if you must write quicksort.
- Reach for counting sort only when the value range is small and known.
- For top-k, use a heap (**14.8**); for the kth element, quickselect (**14.11**).
- For data larger than memory, sort chunks and merge with `heapq.merge`.
- State stability and space when you compare sorts - they are what actually differ.

## Practice Exercises

Try these before moving on.

1. Add median-of-three pivot selection to `quicksort_naive` and re-run the sorted-input case. Does it fix the depth?
2. 🔴 Implement merge sort with `<` instead of `<=` and construct an input that proves it is no longer stable.
3. Implement quicksort **in place** with Lomuto partitioning. What does it save over the three-list version, and what does it cost in readability?
4. Measure `sorted()` on data that is 99%, 90% and 50% sorted. Where does the adaptive advantage disappear?
5. Implement bucket sort for uniformly distributed floats in [0, 1) and compare with `sorted()` at n = 100,000.
6. Use `functools.cmp_to_key` to sort with a three-way comparator, and explain why Python removed `cmp` in the first place.
7. 🔴 Sort 1,000,000 integers in the range 0-255 with counting sort and with `sorted()`. Explain the result in terms of the Ω(n log n) bound.
8. Write a stability checker: sort `(key, index)` pairs by key only, and assert the indices are increasing within each key group.